# M8a · Handling Missing Values

**Goal:** quantify and impute missingness in `marts.v_ml_features_full`
before training. CRISP-DM Phase 3 (Data Preparation): "Clean Data".

**Why this step matters:** scikit-learn estimators reject NaN by default.
A naive `dropna()` on a wide feature matrix can discard 50%+ of rows;
the right strategy is per-column, informed by *why* the value is missing
(MCAR vs MAR vs MNAR).

**Strategies you'll see used here:**

| Pattern                         | Strategy                                      | Why                                                   |
|---------------------------------|-----------------------------------------------|-------------------------------------------------------|
| `avg_fuel_used_l` (94% null)    | Drop the column                               | Cleaning rule C4 nullified almost all of it.          |
| `monthly_idle_ratio` (NaN where ignition_on=0) | Fill with 0                          | Conceptually no-idle when no ignition.               |
| `harsh_*_per_100km` (already 0 in view) | Already imputed (`COALESCE` in SQL)   | The view contract is non-null.                       |
| `avg_speed_over_limit`          | Median by `vehicle_class_enc`                 | Per-class central tendency.                          |
| Numeric tail features           | `SimpleImputer(strategy='median')`            | Robust to outliers we'll handle in 03_handle_outliers|

**Exit criteria:**
- A clean `pandas.DataFrame` with **zero NaN** ready for outlier handling.
- A printed missingness table (before / after) showing what was changed.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — load the unified ML view

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    df = pd.read_sql(text('SELECT * FROM marts.v_ml_features_full'), conn)
print(f'rows={len(df):,}  cols={df.shape[1]}')
df.head(3)


## 3. Diagnose — missingness profile

In [ ]:
import pandas as pd
miss = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
miss = miss[miss > 0].rename('pct_missing').to_frame()
print(f'columns with any missingness: {len(miss)}')
display(miss)


## 4. Strategy — per-column rules

In [ ]:
# 4a. Drop columns that are >90% missing — they carry no signal.
HIGH_NULL_DROP = miss[miss['pct_missing'] > 90].index.tolist()
print('drop:', HIGH_NULL_DROP)
df_clean = df.drop(columns=HIGH_NULL_DROP)

# 4b. Conceptual zero-fills for derived ratios that are NaN only by definition.
ZERO_FILL_IF_NAN = ['monthly_idle_ratio', 'avg_speed_over_limit',
                    'avg_speed_ratio', 'stops_per_trip']
for col in ZERO_FILL_IF_NAN:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# 4c. For remaining numeric columns, impute with median.
from sklearn.impute import SimpleImputer
num_cols = df_clean.select_dtypes(include='number').columns
imputer = SimpleImputer(strategy='median')
df_clean[num_cols] = imputer.fit_transform(df_clean[num_cols])

print('remaining NaN total:', int(df_clean.isna().sum().sum()))


## 5. Verify

In [ ]:
assert df_clean.isna().sum().sum() == 0, 'still has NaN'
print(f'final shape: {df_clean.shape}')

# Persist for the next notebook (handles outliers).
import pathlib
out = pathlib.Path('../../data/ml/step1_no_missing.parquet')
out.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(out, index=False)
print('saved to', out.resolve())
